# Bayesian model performance map

This notebook runs the strict Friday-to-Friday walk-forward evaluator on the Bayesian model only. Each fit sees only matches with `kickoff < cutoff`; the target window is `[cutoff, cutoff + 7 days)`. Stakes use the scenario-based `optimise_portfolio_torch` path: joint P&L scenarios simulated from shared posterior draws, maximising expected log-growth of the bankroll. Change the configuration cell before running.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from footix.data_io.utils_scrapper import add_match_id, to_snake_case
from footix.evaluation import BacktestConfig, bayesian_spec, run_backtest
from footix.strategy.bets import Bet
from footix.strategy.select_bets import select_bets_diagnostics, select_bets_posterior

In [ ]:
DATA_FILE = Path('../data/FRA Ligue 1_2425.csv')
RUN = True

config = BacktestConfig(
    bankroll=1_000.0,
    max_fraction=0.30,
    edge_floor=0.0,
    optimizer_iters=500,
    n_scenarios=10_000,
    markets=('1X2', 'O/U2.5'),
)

## Robust Bayesian selection

Bets are not picked because the model is sure of an outcome, but because the odds look profitable even under posterior uncertainty. `select_bets_posterior` keeps a bet only when its pessimistic edge bound `Q_alpha(odds * p - 1)` is positive, then picks at most one bet per match: the one with the largest robust Kelly fraction `(odds * Q_alpha(p) - 1) / (odds - 1)`, provided it is the best edge of its match in more than `rho_min` of the posterior draws (ambiguity filter). The walk-forward evaluator calls this function internally for the Bayesian spec; the demo below shows the same logic on synthetic posterior samples.

In [ ]:
rng = np.random.default_rng(0)

def posterior_samples(mean, std, n=10_000):
    return np.clip(rng.normal(mean, std, n), 1e-6, 1.0)

# Heavy favourite: high probability, but a negative conservative edge.
favorite = Bet('match1', 'H', 1.35, 0.78)
# Value bet: the 10% lower edge bound is still positive.
value = Bet('match2', 'H', 1.80, 0.64)
# Same match, two near-identical selections: ambiguous, match is skipped.
amb_home = Bet('match3', 'H', 2.0, 0.50)
amb_away = Bet('match3', 'A', 2.0, 0.50)

samples = {
    ('match1', 'H'): posterior_samples(0.78, 0.08),
    ('match2', 'H'): posterior_samples(0.64, 0.03),
    ('match3', 'H'): posterior_samples(0.50, 0.04),
    ('match3', 'A'): posterior_samples(0.50, 0.04),
}

selected = select_bets_posterior(
    [favorite, value, amb_home, amb_away],
    samples,
    alpha=config.select_alpha,
    delta=config.select_delta,
    rho_min=config.select_rho_min,
)
for bet in selected:
    print(bet)

## Selection diagnostics

`select_bets_diagnostics` keeps every candidate and reports why it was accepted or rejected (edge bound, not the best of its match, ambiguity). Run the walk-forward with `staking='flat'` (fixed 1% stake per bet) to evaluate the selection alone before trusting portfolio-optimised results.

In [ ]:
select_bets_diagnostics(
    [favorite, value, amb_home, amb_away],
    samples,
    alpha=config.select_alpha,
    delta=config.select_delta,
    rho_min=config.select_rho_min,
)

In [ ]:
if RUN:
    data = pd.read_csv(DATA_FILE)
    data.columns = [to_snake_case(column) for column in data.columns]
    data = add_match_id(data)
    n_teams = len(set(data['home_team']) | set(data['away_team']))

    specs = [bayesian_spec(n_goals=20, n_teams=n_teams, random_seed=42)]

    result = run_backtest(data, specs, config)
    display(result.windows)
else:
    print('Set RUN = True after checking DATA_FILE.')

In [ ]:
if RUN and not result.predictions.empty:
    summary = (
        result.predictions.groupby(['model', 'market'])[['rps', 'log_loss', 'brier', 'accuracy']]
        .mean()
        .sort_index()
    )
    display(summary)

    for (model, market), values in result.predictions.groupby(['model', 'market']):
        values.groupby('cutoff')['rps'].mean().plot(label=f'{model} {market}')
    plt.legend()
    plt.grid()
    plt.show()

    if not result.bets.empty:
        display(result.bets)
        display(result.windows.groupby('model')[['profit', 'total_stake', 'bankroll_after']].last())

    result.windows.to_csv('bayesian_performance_windows.csv', index=False)
    result.predictions.to_csv('bayesian_performance_predictions.csv', index=False)
    result.bets.to_csv('bayesian_performance_bets.csv', index=False)